This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [1]:
import great_expectations
from great_expectations.core import ExpectationConfiguration
context = great_expectations.get_context()
import logging

In [2]:
import os
gx_context_root_dir=os.environ['GX_CONTEXT_ROOT_DIR']

In [3]:
import yaml

In [4]:
from datetime import date,datetime

In [5]:
logging.basicConfig(level=logging.WARN, force = True)

In [6]:
with open(f"{gx_context_root_dir}/datasources/datasources.yml", "r") as ymlfile:
    datasource_config = yaml.full_load(ymlfile)

In [7]:
datasource_config.get("project")

'gfw-google-827'

In [8]:
gx_project = datasource_config.get("project")
gx_datasource = context.get_datasource(gx_project)

In [9]:
gx_datasource.get_asset_names()

{'all_vessels-1',
 'all_vessels-2',
 'encounters-2.5',
 'encounters-3.0.0',
 'features_-2.5',
 'features_-3.0.0',
 'fishing_score_-2.5',
 'fishing_score_-3.0.0',
 'fragments-3.0.0',
 'loitering-2.5',
 'loitering-3.0.0',
 'messages-2.5',
 'messages-3.0.0',
 'messages_positions-2.5',
 'messages_positions-3.0.0',
 'messages_scored_-2.5',
 'messages_scored_-3.0.0',
 'messages_segmented_-2.5',
 'messages_segmented_-3.0.0',
 'port_visits-2.5',
 'port_visits-3.0.1',
 'satellite_timing_offsets-2.5',
 'satellite_timing_offsets-3.0.0',
 'segment_identity_daily_-2.5',
 'segment_identity_daily_-3.0.0',
 'segment_info-2.5',
 'segment_info-3.0.0',
 'segment_vessel-2.5',
 'segment_vessel-3.0.0',
 'segment_vessel_daily_-2.5',
 'segment_vessel_daily_-3.0.0',
 'segments-2.5',
 'segments-3.0.0',
 'segs_activity-2.5',
 'segs_activity-3.0.0',
 'segs_activity_daily-2.5',
 'segs_activity_daily-3.0.0',
 'ssvids_identities-2.5',
 'ssvids_identities-3.0.0',
 'ssvids_identities_daily-2.5',
 'ssvids_identities_da

In [10]:
qa=gx_datasource.get_asset("vessel_info-2.5")


In [11]:
context.list_expectation_suite_names()

['gfw-google-827.alerts.encounters.2-5',
 'gfw-google-827.alerts.encounters.3-0-0',
 'gfw-google-827.alerts.features_.2-5',
 'gfw-google-827.alerts.features_.3-0-0',
 'gfw-google-827.alerts.fishing_score_.2-5',
 'gfw-google-827.alerts.fishing_score_.3-0-0',
 'gfw-google-827.alerts.fragments.3-0-0',
 'gfw-google-827.alerts.loitering.2-5',
 'gfw-google-827.alerts.loitering.3-0-0',
 'gfw-google-827.alerts.messages.2-5',
 'gfw-google-827.alerts.messages.3-0-0',
 'gfw-google-827.alerts.messages_positions.2-5',
 'gfw-google-827.alerts.messages_positions.3-0-0',
 'gfw-google-827.alerts.messages_scored_.2-5',
 'gfw-google-827.alerts.messages_scored_.3-0-0',
 'gfw-google-827.alerts.messages_segmented_.2-5',
 'gfw-google-827.alerts.messages_segmented_.3-0-0',
 'gfw-google-827.alerts.satellite_timing_offsets.2-5',
 'gfw-google-827.alerts.satellite_timing_offsets.3-0-0',
 'gfw-google-827.alerts.segment_identity_daily_.2-5',
 'gfw-google-827.alerts.segment_identity_daily_.3-0-0',
 'gfw-google-827.a

In [12]:
current_asset_name = 'fragment'
current_asset_constraints=[es for es in context.list_expectation_suite_names() if current_asset_name in es and 'constraints' in es]
current_asset_constraints

['gfw-google-827.constraints.fragments.3-0-0']

In [13]:
current_expectation_suite_name = current_asset_constraints[0]
print(current_expectation_suite_name)
current_expectation_suite=context.get_expectation_suite(current_expectation_suite_name)    
current_expectation_suite_asset_name=current_expectation_suite.meta.get('asset_name')
current_expectation_suite_datasource_name=current_expectation_suite.meta.get('datasource_name')
current_expectation_suite_version_number=current_expectation_suite.meta.get('version_number')

gx_asset=gx_datasource.get_asset(current_expectation_suite_asset_name)
gx_splitter=gx_asset.splitter
if gx_splitter is not None:
    DATE_PARTITION_COLUMN=gx_splitter.column_name
    br_options={DATE_PARTITION_COLUMN: '2023-04-01'}
else:
    br_options={}
gx_br = gx_asset.build_batch_request(br_options)


gfw-google-827.constraints.fragments.3-0-0


SELECT PERCENTILE_CONT(COUNT(*), 0.5) 
FROM (SELECT * FROM tech_great_expectations.v_unfiltered_pipe_ais_v3_alpha_internal_fragments_) 
WHERE SPLITTER_COLUMN > '2023-03-01'
GROUP BY SPLITTER_COLUMN 
ORDER BY SPLITTER_COLUMN DESC 
LIMIT 30


In [15]:

gx_batches = gx_datasource.get_batch_list_from_batch_request(gx_br)


In [16]:

gx_validator = context.get_validator_using_batch_list(current_expectation_suite, gx_batches)



In [28]:
print(gx_validator.get_evaluation_parameter("urn:great_expectations:stores:metric_store:expect_table_row_count_to_be_between.result.observed_value"))

None


In [15]:

gx_validator.expect_table_row_count_to_be_between(
    min_value={"$PARAMETER": f"trunc(urn:great_expectations:stores:custom_metric_store:median_row_count_30_days * 0.8)"},
    max_value={"$PARAMETER": f"trunc(urn:great_expectations:stores:custom_metric_store:median_row_count_30_days * 1.2)"}, 
)

  warnings.warn(



Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

{
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "success": true,
  "result": {
    "observed_value": 360338
  }
}

In [16]:
gx_validator.expect_column_values_to_be_unique('frag_id')

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

{
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "success": true,
  "result": {
    "element_count": 360338,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  }
}

In [17]:
gx_validator.expect_column_min_to_be_between('msg_count', min_value=0, strict_min=True)

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

{
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "success": true,
  "result": {
    "observed_value": 1
  }
}

In [26]:
gx_validator.expect_queried_column_values_to_exist_in_second_table_column(
    template_dict={
        "second_table_full_name": "pipe_ais_v3_alpha_internal.segments_20230401",
        "first_table_column": "frag_id",
        "second_table_column": "frag_id",
        "condition": "1=1"
    }
)

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "success": true,
  "result": {
    "Rows with IDs in first table missing in second table": 0
  }
}

In [41]:
timestamp_from_id_sql = f"""
((
    SELECT 
    TIMESTAMP(STRING_AGG(arr, '-'))
    FROM UNNEST(SPLIT(frag_id, '-')) AS arr WITH OFFSET as offset
    WHERE offset BETWEEN 1 and 3
))
"""

In [20]:
gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
    SELECT *
    FROM {{active_batch}}
    WHERE {timestamp_from_id_sql('frag_id')} != first_msg_timestamp
"""}, value=0, meta={
            "notes": {
                "format": "markdown",
                "content": "The `frag_id` timestamp should be based on the `first_msg_timestamp` of a fragment.",
        }
})

  warnings.warn(str(e), UserWarning)

  warnings.warn(



Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "result": {
    "observed_value": 0
  },
  "meta": {},
  "success": true
}

In [22]:
gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
    SELECT *
    FROM {{active_batch}}
    WHERE DATE({timestamp_from_id_sql('frag_id')}) != DATE(timestamp)
"""}, value=0, meta={
            "notes": {
                "format": "markdown",
                "content": "All timestamps in a fragment should have the same date as the `frag_id`'s timestamp.",
        }
})

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "result": {
    "observed_value": 0
  },
  "meta": {},
  "success": true
}

In [23]:
gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
    SELECT *
    FROM {{active_batch}}
    WHERE DATE({timestamp_from_id_sql('frag_id')}) != DATE(last_msg_timestamp)
"""}, value=0, meta={
            "notes": {
                "format": "markdown",
                "content": "The date of the `frag_id`'s timestamp should match the date of the `last_msg_timestamp`.",
        }
})

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "result": {
    "observed_value": 0
  },
  "meta": {},
  "success": true
}

In [24]:
gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
    SELECT *
    FROM {{active_batch}}
    WHERE LEFT(frag_id, STRPOS(frag_id, "-")-1) != ssvid
"""}, value=0, meta={
            "notes": {
                "format": "markdown",
                "content": "The `frag_id` should start with the `ssvid`.",
        }
})

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "result": {
    "observed_value": 0
  },
  "meta": {},
  "success": true
}

In [47]:
gx_validator.expect_column_values_to_be_unique('frag_id')
gx_validator.expect_column_values_to_not_be_null('frag_id')


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 360338,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [21]:
gx_validator.save_expectation_suite(discard_failed_expectations=False)